In [2]:
import os
import sys

import langchain
import langchain_core
import langchain_openai

from dotenv import load_dotenv

load_dotenv()

# 가상환경 확인
print(f"Python 실행 경로: {sys.executable}")
if ".venv" in sys.executable:
    
    print("[OK] .venv 가상환경 사용 중")
else:
    print("[WARN] .venv 가상환경이 아닌 인터프리터가 선택되어 있습니다")

print("[OK] OpenAI API 키 설정됨" if os.getenv("OPENAI_API_KEY") else "[ERROR] OpenAI API 키 미설정")
print("[OK] Anthropic API 키 설정됨" if os.getenv("ANTHROPIC_API_KEY") else "[ERROR] Anthropic API 키 미설정")
print("[OK] Google API 키 설정됨" if os.getenv("GOOGLE_API_KEY") else "[ERROR] Google API 키 미설정")
print("[OK] LangSmith 키 설정됨" if os.getenv("LANGSMITH_API_KEY") else "[ERROR] LangSmith 키 미설정")

print(f"LangChain: {langchain.__version__}")
print(f"LangChain Core: {langchain_core.__version__}")
print(f"LangChain OpenAI: {langchain_openai.__version__}")

Python 실행 경로: d:\yugyeong\hanwha_0902\days\ex_0918\.venv\Scripts\python.exe
[OK] .venv 가상환경 사용 중
[OK] OpenAI API 키 설정됨
[ERROR] Anthropic API 키 미설정
[ERROR] Google API 키 미설정
[OK] LangSmith 키 설정됨
LangChain: 1.4.1
LangChain Core: 1.6.3
LangChain OpenAI: 1.6.2


In [ ]:
import faiss

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_core.documents import Document

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

embedding_size = 1536

# IndexFlatL2 : 벡터 사이의 거리를 계산해서 서로 가까운 벡터를 찾는 방식
index = faiss.IndexFlatL2(embedding_size)

vectorstore = FAISS(
    embedding_function = embeddings_model,
    index = index,
    docstore = InMemoryDocstore(),
    index_to_docstore_id =
    {}
)

In [7]:
conversations = [
    Document(
        page_content=(
            "Human: 안녕하세요, 오늘 면접에 참석해주셔서 감사합니다. "
            "자기소개 부탁드립니다.\n"
            "AI: 안녕하세요. 저는 컴퓨터 과학을 전공한 신입 개발자입니다. "
            "대학에서는 주로 자바와 파이썬을 사용했으며, 최근에는 웹 개발 "
            "프로젝트에 참여하여 실제 사용자를 위한 서비스를 개발했습니다."
        )
    ),

    Document(
        page_content=(
            "Human: 프로젝트에서 어떤 역할을 맡았나요?\n"
            "AI: 제가 맡은 역할은 백엔드 개발자였습니다. "
            "사용자 데이터 처리와 서버 로직 개발을 담당했으며, "
            "RESTful API를 구현하여 프론트엔드와 통신했습니다. "
            "또한 데이터베이스 설계에도 참여했습니다."
        )
    ),

    Document(
        page_content=(
            "Human: 팀 프로젝트에서 어려움을 겪었던 경험이 있다면 어떻게 해결했나요?\n"
            "AI: 프로젝트 초기에 의사소통 문제로 어려움이 있었습니다. "
            "이를 해결하기 위해 정기적인 미팅을 갖고 각자의 진행 상황을 공유했습니다."
        )
    ),

    Document(
        page_content=(
            "Human: 개발자로서 자신의 강점은 무엇이라고 생각하나요?\n"
            "AI: 제 강점은 빠른 학습 능력과 문제 해결 능력입니다. "
            "새로운 기술이나 도구를 빠르게 습득할 수 있으며, "
            "복잡한 문제에 직면했을 때 해결책을 제시할 수 있습니다."
        )
    ),
]


In [ ]:
# Document > Embedding > 숫자 벡터 > FAISS 및 Docstore에 저장
vectorstore.add_documents(conversations)

# 리트리버 : 질문과 관련된 문서를 찾아오는 검색기
retriever = vectorstore.as_retriever(
    search_kwargs={"k" : 1}
)

In [9]:
query = "면접자 전공은 무엇인가요?"

docs = retriever.invoke(query)

for doc in docs:
    print(doc.page_content)

Human: 안녕하세요, 오늘 면접에 참석해주셔서 감사합니다. 자기소개 부탁드립니다.
AI: 안녕하세요. 저는 컴퓨터 과학을 전공한 신입 개발자입니다. 대학에서는 주로 자바와 파이썬을 사용했으며, 최근에는 웹 개발 프로젝트에 참여하여 실제 사용자를 위한 서비스를 개발했습니다.


In [10]:
query = "면접자가 프로젝트에서 맡은 역할은 무엇인가요?"

docs = retriever.invoke(query)

for doc in docs:
    print(doc.page_content)

Human: 프로젝트에서 어떤 역할을 맡았나요?
AI: 제가 맡은 역할은 백엔드 개발자였습니다. 사용자 데이터 처리와 서버 로직 개발을 담당했으며, RESTful API를 구현하여 프론트엔드와 통신했습니다. 또한 데이터베이스 설계에도 참여했습니다.
